# Open-Meteo Training Pipeline

Runs the full Open-Meteo branch pipeline: optional weather download, input validation, `main.py` training, and W&B logging.

## 1. Setup

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
os.environ['PYTHONPATH'] = str(REPO_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
PYTHON = sys.executable

def run(cmd, env=None):
    cmd = [str(c) for c in cmd]
    print('$ ' + ' '.join(cmd))
    return subprocess.run(cmd, cwd=REPO_ROOT, env=env, check=True)

print('repo root:', REPO_ROOT)
print('python   :', PYTHON)

## 2. Configuration

In [ ]:
DATA_YEAR = 2019
START_DATE = '2019-03-01'
END_DATE = '2019-12-31'
WEATHER_SOURCE = 'openmeteo_historical_forecast'
FEATURE_SET = 'openmeteo_operational'

OPENMETEO_PATH = Path(f'data/openmeteo_piedmont_{DATA_YEAR}.nc')
PLANT_MAPPING_PATH = Path('data/plant_mapping.csv')
ENERGY_COORDS_PATH = Path('data/energy_with_coordinates.csv')
PLANTS_PATH = ENERGY_COORDS_PATH
SENTINEL_DIR = Path('/data/SentinelPV/energy_data/piemonte_energy_data/single_ups')

SEEDS = [42]
N_EPOCHS = 15
MAX_STEPS_PER_EPOCH = None
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 1e-4
SEQ_LEN = 24
PATCH_LEN = 4
STRIDE = 2
BILSTM_POOLING = 'attn'

PEAK_ALPHA = 2.5
PEAK_GAMMA = 2.0
PEAK_LOSS_WEIGHT = 0.25
UNDER_PENALTY = 3.0
ETA_MAX = 0.98
CALIBRATION_KPI = 'none'
QS_LOSS_WEIGHTING = False
QS_LOSS_FLOOR = 0.2
APPLY_OUTLIER_FILTER = False
OUTLIER_QS_DAYTIME_THRESHOLD = 0.30
OUTLIER_MIN_VALID_DAYTIME = 200

CHECKPOINT_DIR_BASE = f'checkpoints/openmeteo_seq{SEQ_LEN}'

WANDB_MODE = 'online'
WANDB_PROJECT = 'PhysiQ-PV'
WANDB_ENTITY = 'albertopedalino-politecnico-di-torino'
WANDB_RUN_NAME = (
    '{feature_set}_seq{seq_len}_peakw{peak_loss_weight:g}'
    '_pool{bilstm_pooling}{quality_suffix}_seed{seed}'
)
LOSS_TAG = 'qs_weighted_loss' if QS_LOSS_WEIGHTING else 'unweighted_loss'
WANDB_TAGS = [
    'openmeteo', 'st-gnn', 'mc-dropout-ready',
    f'seq_len_{SEQ_LEN}', LOSS_TAG,
]

RUN_DOWNLOAD_IF_MISSING = True
RUN_PIPELINE_CHECK = True
RUN_TRAINING = True

env = dict(os.environ)
env.update({
    'DATA_YEAR': str(DATA_YEAR),
    'WEATHER_SOURCE': WEATHER_SOURCE,
    'FEATURE_SET': FEATURE_SET,
    'OPENMETEO_PATH': str(OPENMETEO_PATH),
    'SENTINEL_DIR': str(SENTINEL_DIR),
    'PLANT_MAPPING_PATH': str(PLANT_MAPPING_PATH),
    'ENERGY_COORDS_PATH': str(ENERGY_COORDS_PATH),
    'SEEDS': ','.join(str(s) for s in SEEDS),
    'N_EPOCHS': str(N_EPOCHS),
    'MAX_STEPS_PER_EPOCH': '' if MAX_STEPS_PER_EPOCH is None else str(MAX_STEPS_PER_EPOCH),
    'EARLY_STOPPING_PATIENCE': str(EARLY_STOPPING_PATIENCE),
    'EARLY_STOPPING_MIN_DELTA': str(EARLY_STOPPING_MIN_DELTA),
    'SEQ_LEN': str(SEQ_LEN),
    'PATCH_LEN': str(PATCH_LEN),
    'STRIDE': str(STRIDE),
    'BILSTM_POOLING': BILSTM_POOLING,
    'PEAK_ALPHA': str(PEAK_ALPHA),
    'PEAK_GAMMA': str(PEAK_GAMMA),
    'PEAK_LOSS_WEIGHT': str(PEAK_LOSS_WEIGHT),
    'UNDER_PENALTY': str(UNDER_PENALTY),
    'ETA_MAX': str(ETA_MAX),
    'CALIBRATION_KPI': CALIBRATION_KPI,
    'QS_LOSS_WEIGHTING': str(int(QS_LOSS_WEIGHTING)),
    'QS_LOSS_FLOOR': str(QS_LOSS_FLOOR),
    'APPLY_OUTLIER_FILTER': str(int(APPLY_OUTLIER_FILTER)),
    'OUTLIER_QS_DAYTIME_THRESHOLD': str(OUTLIER_QS_DAYTIME_THRESHOLD),
    'OUTLIER_MIN_VALID_DAYTIME': str(OUTLIER_MIN_VALID_DAYTIME),
    'CHECKPOINT_DIR_BASE': CHECKPOINT_DIR_BASE,
    'WANDB_MODE': WANDB_MODE,
    'WANDB_PROJECT': WANDB_PROJECT,
    'WANDB_ENTITY': WANDB_ENTITY,
    'WANDB_RUN_NAME': WANDB_RUN_NAME,
    'WANDB_TAGS': ','.join(WANDB_TAGS),
})

config_keys = [
    'DATA_YEAR', 'WEATHER_SOURCE', 'FEATURE_SET', 'OPENMETEO_PATH',
    'SENTINEL_DIR', 'SEEDS', 'N_EPOCHS', 'SEQ_LEN', 'PATCH_LEN', 'STRIDE',
    'BILSTM_POOLING', 'PEAK_LOSS_WEIGHT', 'QS_LOSS_WEIGHTING',
    'CHECKPOINT_DIR_BASE', 'WANDB_MODE', 'WANDB_PROJECT', 'WANDB_ENTITY',
    'WANDB_RUN_NAME', 'WANDB_TAGS',
]
print(json.dumps({k: env[k] for k in config_keys}, indent=2))

## 3. Inputs

In [ ]:
checks = {
    'Sentinel dir': SENTINEL_DIR.exists(),
    'plant coordinates csv': PLANTS_PATH.exists(),
    'Open-Meteo NetCDF': OPENMETEO_PATH.exists(),
}
for name, ok in checks.items():
    print(('OK     ' if ok else 'MISSING') + '  ' + name)

if not SENTINEL_DIR.exists():
    raise FileNotFoundError(f'Sentinel directory not found: {SENTINEL_DIR}')
if not PLANTS_PATH.exists():
    raise FileNotFoundError(f'Plant coordinate file not found: {PLANTS_PATH}')

## 4. Download Open-Meteo If Needed

In [ ]:
if RUN_DOWNLOAD_IF_MISSING and not OPENMETEO_PATH.exists():
    run([
        PYTHON, 'scripts/download_openmeteo_historical_forecast.py',
        '--plants-path', PLANTS_PATH,
        '--start-date', START_DATE,
        '--end-date', END_DATE,
        '--out', OPENMETEO_PATH,
        '--source', 'historical_forecast',
    ])
else:
    print('Open-Meteo download skipped; file exists or RUN_DOWNLOAD_IF_MISSING=False.')

if not OPENMETEO_PATH.exists():
    raise FileNotFoundError(f'Open-Meteo NetCDF not found: {OPENMETEO_PATH}')

## 5. Pipeline Check

In [ ]:
if RUN_PIPELINE_CHECK:
    run([
        PYTHON, 'scripts/check_openmeteo_pipeline.py',
        '--openmeteo-path', OPENMETEO_PATH,
        '--sentinel-dir', SENTINEL_DIR,
        '--plant-mapping', PLANT_MAPPING_PATH,
        '--energy-coords', ENERGY_COORDS_PATH,
        '--max-plants', '5',
        '--max-time-steps', '200',
    ])
else:
    print('Pipeline check skipped.')

## 6. W&B

In [ ]:
import wandb
print('wandb version:', wandb.__version__)
print('WANDB_MODE   :', env['WANDB_MODE'])
print('project      :', env['WANDB_PROJECT'])
print('entity       :', env['WANDB_ENTITY'])
print('If this cell fails later at training init, run `wandb login` in the same environment.')

## 7. Train And Log To W&B

In [ ]:
if RUN_TRAINING:
    run([PYTHON, 'main.py'], env=env)
else:
    print('Training skipped. Set RUN_TRAINING=True to execute main.py.')

## 8. Outputs

In [ ]:
checkpoint_dirs = sorted(Path('checkpoints').glob(f'{Path(CHECKPOINT_DIR_BASE).name}_pool*seed*'))
print('checkpoint dirs:', len(checkpoint_dirs))
for path in checkpoint_dirs:
    summary = path / 'loss_history.json'
    if summary.exists():
        data = json.loads(summary.read_text())
        best = min(data.get('val', [float('nan')]))
        print(f'{path}  best_val={best:.4f}')
    else:
        print(path)